# 通用模型数据生成（宿主机）

支持的模型：`mlp` (784→128→10) / `lenet5` (Conv+Pool+FC)

修改下方 `ARCH` 即可切换模型，其余流程完全一致。

In [1]:
import sys
sys.path.insert(0, '.')
from model_zoo import *

## 1. 选择模型架构

In [2]:
# ===== 在这里选择模型 =====
ARCH = 'lenet5'   # 'mlp' or 'lenet5'
# ==========================

SEED = 42
EPOCHS = {'mlp': 10, 'lenet5': 15}[ARCH]
NUM_TEST = 20
OUTPUT_DIR = f'{ARCH}_data'
MODEL_PATH = f'{ARCH}.pt'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Arch: {ARCH}, Device: {device}')

Arch: lenet5, Device: cuda


## 2. 训练

In [3]:
model = build_model(ARCH)
float_acc = train(model, epochs=EPOCHS, device=device, seed=SEED)
torch.save(model.state_dict(), MODEL_PATH)
print(f'Saved to {MODEL_PATH}')

100.0%
100.0%
100.0%
100.0%


Training lenet5 for 15 epochs (seed=42)
  Epoch 1/15: loss=0.4440, acc=96.16%
  Epoch 2/15: loss=0.1109, acc=97.55%
  Epoch 3/15: loss=0.0781, acc=98.10%
  Epoch 4/15: loss=0.0610, acc=98.23%
  Epoch 5/15: loss=0.0506, acc=98.71%
  Epoch 6/15: loss=0.0434, acc=98.56%
  Epoch 7/15: loss=0.0377, acc=98.90%
  Epoch 8/15: loss=0.0331, acc=98.58%
  Epoch 9/15: loss=0.0304, acc=98.76%
  Epoch 10/15: loss=0.0260, acc=98.69%
  Epoch 11/15: loss=0.0253, acc=98.72%
  Epoch 12/15: loss=0.0218, acc=98.72%
  Epoch 13/15: loss=0.0201, acc=98.86%
  Epoch 14/15: loss=0.0176, acc=98.87%
  Epoch 15/15: loss=0.0163, acc=98.82%
Saved to lenet5.pt


In [4]:
# 或加载已有模型
# model = build_model(ARCH)
# model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
# model = model.to(device).eval()

## 3. 量化

In [5]:
qparams = quantize(model.to(device).eval(), device)


Quantizing lenet5...
  conv1: s_w=0.004462, s_out=0.018894, mult=61, shift=16
  conv2: s_w=0.004358, s_out=0.055850, mult=97, shift=16
  fc3: s_w=0.004595, s_out=0.123084, mult=137, shift=16
  fc4: s_w=0.004728, s_out=0.151166, mult=252, shift=16
  fc5: s_w=0.004400, s_out=0.277012, mult=157, shift=16


## 4. INT8 准确率验证

In [6]:
transform_raw = transforms.ToTensor()
test_set = datasets.MNIST('./data', train=False, download=True, transform=transform_raw)

correct_i8, correct_fp, total = 0, 0, len(test_set)
for i in range(total):
    img, label = test_set[i]
    img_u8 = np.clip(np.round(img.numpy().flatten() * 255), 0, 255).astype(np.uint8)
    pred_i8, _, _ = int8_infer(img_u8, qparams)
    if pred_i8 == label: correct_i8 += 1
    with torch.no_grad():
        pred_fp = model(img.unsqueeze(0).to(device)).argmax(1).item()
    if pred_fp == label: correct_fp += 1
    print(f"now, epoch {i} in {total}.")

print(f'Float32: {100.*correct_fp/total:.2f}% ({correct_fp}/{total})')
print(f'INT8:    {100.*correct_i8/total:.2f}% ({correct_i8}/{total})')
print(f'Drop:    {100.*(correct_fp-correct_i8)/total:.2f}%')

Float32: 98.82% (9882/10000)
INT8:    98.81% (9881/10000)
Drop:    0.01%


## 5. 导出 Hex

In [7]:
test_images = []
test_labels = []
for i in range(NUM_TEST):
    img, label = test_set[i]
    test_images.append(np.clip(np.round(img.numpy().flatten()*255), 0, 255).astype(np.uint8))
    test_labels.append(int(label))

export_hex(qparams, OUTPUT_DIR, test_images, test_labels, NUM_TEST)

  [0000] label=7, pred=7 ✓
  [0001] label=2, pred=2 ✓
  [0002] label=1, pred=1 ✓
  [0003] label=0, pred=0 ✓
  [0004] label=4, pred=4 ✓
  [0005] label=1, pred=1 ✓
  [0006] label=4, pred=4 ✓
  [0007] label=9, pred=9 ✓
  [0008] label=5, pred=5 ✓
  [0009] label=9, pred=9 ✓
  [0010] label=0, pred=0 ✓
  [0011] label=6, pred=6 ✓
  [0012] label=9, pred=9 ✓
  [0013] label=0, pred=0 ✓
  [0014] label=1, pred=1 ✓
  [0015] label=5, pred=5 ✓
  [0016] label=9, pred=9 ✓
  [0017] label=7, pred=7 ✓
  [0018] label=3, pred=8 ✗
  [0019] label=4, pred=4 ✓

  INT8 accuracy: 95.0% (19/20)

Exported to lenet5_data/


'lenet5_data'

In [8]:
import shutil
shutil.make_archive(OUTPUT_DIR, 'zip', '.', OUTPUT_DIR)
print(f'Created {OUTPUT_DIR}.zip — upload to PYNQ')

Created lenet5_data.zip — upload to PYNQ
